In [1]:
!pip -q install --upgrade pip
!pip -q install tqdm matplotlib pillow opencv-python

try:
    import pycocotools  # noqa
    print('pycocotools already available')
except Exception:
    !pip -q install pycocotools
    try:
        import pycocotools  # noqa
        print('pycocotools installed')
    except Exception:
        print('pycocotools kurulamadı. Windows için alternatif:')
        print('  !pip install pycocotools-windows')

ERROR: To modify pip, please run the following command:
d:\SAYZEK\Sayzek_models\.venv\Scripts\python.exe -m pip -q install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


pycocotools already available



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import random
import time

import numpy as np
import torch
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
from PIL import Image

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

import torchvision
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as F

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


## 1) Dataset yolları

In [ ]:
# Dataset yolu (sizin konum)
DATA_ROOT = r'D:\\SAYZEK\\SAYZEK_COCO'

# Bu dataset yapısında yalnızca train klasörü ve train/_annotations.coco.json var.
# Val klasörü yoksa train içinden otomatik train/val split yapıp COCO json üretiriz.

import json

TRAIN_IMG_DIR = os.path.join(DATA_ROOT, 'train')
VAL_IMG_DIR = os.path.join(DATA_ROOT, 'val')  # varsa kullanılır; yoksa train ile aynı olacak

# Klasik COCO export yolları (train/val klasörleri + annotations/..)
TRAIN_ANN = os.path.join(DATA_ROOT, 'annotations', 'instances_train.json')
VAL_ANN = os.path.join(DATA_ROOT, 'annotations', 'instances_val.json')

# RoboFlow COCO export (tek klasör) yolu
ROBOFLOW_ANN = os.path.join(TRAIN_IMG_DIR, '_annotations.coco.json')
VAL_SPLIT = 0.2
SPLIT_SEED = 42

def _split_coco(ann_path: str, out_train_path: str, out_val_path: str, val_ratio: float = 0.2, seed: int = 42):
    with open(ann_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    images = list(data.get('images', []))
    rng = random.Random(seed)
    rng.shuffle(images)
    n_val = max(1, int(len(images) * val_ratio)) if len(images) > 1 else 0
    val_images = images[:n_val]
    train_images = images[n_val:]
    val_ids = {im['id'] for im in val_images}
    train_ids = {im['id'] for im in train_images}
    anns = list(data.get('annotations', []))
    val_anns = [a for a in anns if a.get('image_id') in val_ids]
    train_anns = [a for a in anns if a.get('image_id') in train_ids]

    def _write(out_path: str, imgs, annotations):
        out = dict(data)
        out['images'] = imgs
        out['annotations'] = annotations
        with open(out_path, 'w', encoding='utf-8') as wf:
            json.dump(out, wf)

    _write(out_train_path, train_images, train_anns)
    _write(out_val_path, val_images, val_anns)

# Yolları çözümle
if os.path.exists(TRAIN_ANN) and os.path.exists(VAL_ANN) and os.path.exists(TRAIN_IMG_DIR) and os.path.exists(VAL_IMG_DIR):
    print('Klasik COCO yapısı bulundu (train/val + annotations).')
else:
    assert os.path.exists(TRAIN_IMG_DIR), f'Train image klasörü bulunamadı: {TRAIN_IMG_DIR}'
    assert os.path.exists(ROBOFLOW_ANN), f'COCO json bulunamadı: {ROBOFLOW_ANN}'
    VAL_IMG_DIR = TRAIN_IMG_DIR
    split_dir = os.path.join(DATA_ROOT, '_splits')
    os.makedirs(split_dir, exist_ok=True)
    TRAIN_ANN = os.path.join(split_dir, 'instances_train.json')
    VAL_ANN = os.path.join(split_dir, 'instances_val.json')
    if not (os.path.exists(TRAIN_ANN) and os.path.exists(VAL_ANN)):
        _split_coco(ROBOFLOW_ANN, TRAIN_ANN, VAL_ANN, val_ratio=VAL_SPLIT, seed=SPLIT_SEED)
    print('RoboFlow COCO yapısı bulundu. Split dosyaları:', TRAIN_ANN, VAL_ANN)

for p in [TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_ANN, VAL_ANN]:
    print(p, '->', 'OK' if os.path.exists(p) else 'MISSING')

assert os.path.exists(TRAIN_ANN) and os.path.exists(VAL_ANN), 'Annotation dosyaları bulunamadı'
assert os.path.exists(TRAIN_IMG_DIR) and os.path.exists(VAL_IMG_DIR), 'Image klasörleri bulunamadı'


RoboFlow COCO yapısı bulundu. Split dosyaları: D:\\SAYZEK\\SAYZEK_COCO\_splits\instances_train.json D:\\SAYZEK\\SAYZEK_COCO\_splits\instances_val.json
D:\\SAYZEK\\SAYZEK_COCO\train -> OK
D:\\SAYZEK\\SAYZEK_COCO\train -> OK
D:\\SAYZEK\\SAYZEK_COCO\_splits\instances_train.json -> OK
D:\\SAYZEK\\SAYZEK_COCO\_splits\instances_val.json -> OK


In [4]:
coco_train = COCO(TRAIN_ANN)
coco_val = COCO(VAL_ANN)

cat_ids = coco_train.getCatIds()
cats = coco_train.loadCats(cat_ids)
cats = sorted(cats, key=lambda x: x['id'])
id2name = {c['id']: c['name'] for c in cats}
cat_id_list = [c['id'] for c in cats]
cat_id_to_contig = {cid: i + 1 for i, cid in enumerate(cat_id_list)}  # 0 background
contig_to_cat_id = {v: k for k, v in cat_id_to_contig.items()}

num_classes = len(cat_id_list) + 1
print('num_classes (incl background):', num_classes)

loading annotations into memory...
Done (t=0.14s)
creating index...
index created!
loading annotations into memory...
Done (t=0.12s)
creating index...
index created!
num_classes (incl background): 5


In [5]:
# Dataset: CocoDetection -> FasterRCNN target dict
class FasterCocoDataset(CocoDetection):
    def __init__(self, img_folder, ann_file, train: bool):
        super().__init__(img_folder, ann_file)
        self.train = train

    def __getitem__(self, idx):
        img, anns = super().__getitem__(idx)
        image_id = self.ids[idx]

        # PIL -> tensor
        img_t = F.to_tensor(img)

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for a in anns:
            # bbox: [x,y,w,h] -> [x1,y1,x2,y2]
            x, y, w, h = a['bbox']
            if w <= 1 or h <= 1:
                continue
            boxes.append([x, y, x + w, y + h])
            cid = int(a['category_id'])
            labels.append(cat_id_to_contig.get(cid, 0))
            areas.append(float(a.get('area', w * h)))
            iscrowd.append(int(a.get('iscrowd', 0)))

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([image_id]),
            'area': areas,
            'iscrowd': iscrowd
        }

        # basit augment: horizontal flip
        if self.train and random.random() < 0.5 and boxes.shape[0] > 0:
            _, h_img, w_img = img_t.shape
            img_t = torch.flip(img_t, dims=[2])
            x1 = boxes[:, 0].clone()
            x2 = boxes[:, 2].clone()
            boxes[:, 0] = w_img - x2
            boxes[:, 2] = w_img - x1
            target['boxes'] = boxes

        return img_t, target

def collate_fn(batch):
    return tuple(zip(*batch))

train_ds = FasterCocoDataset(TRAIN_IMG_DIR, TRAIN_ANN, train=True)
val_ds = FasterCocoDataset(VAL_IMG_DIR, VAL_ANN, train=False)

BATCH_SIZE = 2
NUM_WORKERS = 0

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)

print('train batches:', len(train_loader), 'val batches:', len(val_loader))

loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
train batches: 3595 val batches: 1797


In [ ]:
import torch
import math
import sys
import time
import numpy as np
from tqdm.auto import tqdm
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from pycocotools.cocoeval import COCOeval

# EARLY STOPPING
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, path='best_faster_rcnn.pth'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_map_max = -np.inf
        self.path = path

    def __call__(self, val_map, model):
        score = val_map

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_map, model)
        elif score < self.best_score:
            # Skor artmazsa sayacı arttır
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping sayacı: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            # Skor arttı, kaydet ve sayacı sıfırla
            self.best_score = score
            self.save_checkpoint(val_map, model)
            self.counter = 0

    def save_checkpoint(self, val_map, model):
        if self.verbose:
            print(f'mAP yükseldi ({self.val_map_max:.4f} --> {val_map:.4f}). Model kaydediliyor...')
        torch.save(model.state_dict(), self.path)
        self.val_map_max = val_map

# EVALUATION
@torch.no_grad()
def evaluate_coco(model, data_loader, device, coco_gt):
    model.eval()
    results = []
    
    for images, targets in tqdm(data_loader, desc="Evaluation"):
        images = list(img.to(device) for img in images)
        
        
        outputs = model(images)
        
        # Sonuçları CPU'ya ve COCO formatına al
        for i, output in enumerate(outputs):
            image_id = targets[i]["image_id"].item()
            boxes = output["boxes"].cpu().numpy()
            scores = output["scores"].cpu().numpy()
            labels = output["labels"].cpu().numpy()
            
            for box, score, label in zip(boxes, scores, labels):
                # Faster R-CNN box: x1, y1, x2, y2 -> COCO box: x, y, w, h
                x, y, w, h = box[0], box[1], box[2] - box[0], box[3] - box[1]
                
                # Label ID'yi gerçek COCO ID'sine çevir
                real_cat_id = contig_to_cat_id.get(label, label)
                
                results.append({
                    "image_id": image_id,
                    "category_id": real_cat_id,
                    "bbox": [x, y, w, h],
                    "score": float(score)
                })

    if len(results) == 0:
        print("Model validasyon setinde hiç nesne bulamadı.")
        return 0.0

    # COCO Eval Çalıştır
    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    
    # AP @ IoU=0.50:0.95 değerini döndür (stats[0])
    return coco_eval.stats[0]

# --- 3. MODEL VE OPTIMIZER KURULUMU ---
print("Model hazırlanıyor...")
model = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')

# ROI Head kısmını kendi sınıf sayımıza göre güncelle
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.to(device)

# Ayarlar
LR = 5e-5 # Faster R-CNN için 1e-4 veya 5e-5 genelde iyidir (büyük gelirse düşürürüz)
WEIGHT_DECAY = 1e-4
EPOCHS = 30 # Early stopping olduğu için yüksek verebiliriz

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

# LR Scheduler (Opsiyonel ama önerilir: Platonun aşılmasına yardımcı olur)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
early_stopping = EarlyStopping(patience=5, verbose=True, path='best_faster_rcnn_model.pth')

# --- 4. EĞİTİM DÖNGÜSÜ ---
print(f"Eğitim başlıyor... Hedef: {EPOCHS} Epoch (Patience: 5)")
loss_history = []
map_history = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch")
    
    for images, targets in progress_bar:
        # Veriyi Cihaza Taşı
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            loss_dict = model(images, targets)
            # Faster R-CNN birden fazla loss döndürür, hepsini toplarız
            losses = sum(loss for loss in loss_dict.values())
        
        # Loss değeri sonsuz veya NaN ise durdurma, atla (Güvenlik)
        loss_value = losses.item()
        if not math.isfinite(loss_value):
            print(f"Loss is {loss_value}, skipping batch")
            continue
            
        scaler.scale(losses).backward()
        
        # Gradient Clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss_value
        progress_bar.set_postfix({"loss": f"{loss_value:.4f}"})
    
    # Ortalama Eğitim Kaybı
    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1} Train Loss: {avg_loss:.4f}")
    
    # --- VALIDATION & mAP ---
    # Her epoch sonu mAP hesapla
    current_map = evaluate_coco(model, val_loader, device, coco_val)
    map_history.append(current_map)
    
    # LR Scheduler Güncelle (mAP'ye göre)
    lr_scheduler.step(current_map)
    
    # --- EARLY STOPPING KONTROLÜ ---
    early_stopping(current_map, model)
    
    if early_stopping.early_stop:
        print("Early stopping tetiklendi. Eğitim durduruluyor.")
        break

print("Eğitim tamamlandı.")
# En iyi modeli yükle (hafızadaki model son epoch olabilir, en iyisine dönelim)
model.load_state_dict(torch.load('best_faster_rcnn_model.pth'))
print("En iyi model (best_faster_rcnn_model.pth) yüklendi.")

Model hazırlanıyor...


d:\SAYZEK\Sayzek_models\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Eğitim başlıyor... Hedef: 30 Epoch (Patience: 5)


Epoch 1/30: 100%|██████████| 3595/3595 [20:04<00:00,  2.99batch/s, loss=0.0578]


Epoch 1 Train Loss: 0.2079


...

Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=2.09s).
Accumulating evaluation results...
DONE (t=0.17s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.526
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.914
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.551
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.471
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.578
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.646
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.395
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.538
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.615
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

Epoch 2/30: 100%|██████████| 3595/3595 [20:09<00:00,  2.97batch/s, loss=0.0772]


Epoch 2 Train Loss: 0.1606


Evaluation: 100%|██████████| 1797/1797 [03:47<00:00,  7.90it/s]


Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*


In [ ]:
import matplotlib.pyplot as plt

# Eğitim bloğunda doldurulan 'loss_history' ve 'map_history' listelerini kullanıyoruz
if 'loss_history' in globals() and len(loss_history) > 0:
    epochs = list(range(1, len(loss_history) + 1))

    plt.figure(figsize=(14, 5))

    # 1. Grafik: Training Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss_history, 'b-o', label='Train Loss')
    plt.title('Eğitim Kaybı (Training Loss)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()

    # 2. Grafik: Validation mAP
    plt.subplot(1, 2, 2)
    plt.plot(epochs, map_history, 'r-s', label='Validation mAP (IoU=0.50:0.95)')
    plt.title('Genel Başarım (mAP)')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()
else:
    print("Henüz çizilecek veri yok (Eğitim tamamlanmamış veya history listeleri boş).")

In [ ]:
import torch
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from torchvision.transforms import functional as F

# Ayarlar
NUM_IMAGES = 10
SCORE_THRESHOLD = 0.5  # %50'nin altındaki tahminleri çizme

# Modeli Değerlendirme Moduna Al
model.eval()

# Val setinden rastgele resimler seç
all_img_ids = coco_val.getImgIds()
selected_ids = random.sample(all_img_ids, min(NUM_IMAGES, len(all_img_ids)))

plt.figure(figsize=(15, 6 * NUM_IMAGES))

print(f"Toplam {len(selected_ids)} resim üzerinde tahmin yapılıyor...")

for idx, img_id in enumerate(selected_ids):
    # 1. Resmi Yükle
    img_info = coco_val.loadImgs([img_id])[0]
    img_path = os.path.join(VAL_IMG_DIR, img_info['file_name'])
    
    try:
        original_img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Resim okunamadı: {img_path}")
        continue

    # 2. Modele Ver (Tensor Dönüşümü)
    img_tensor = F.to_tensor(original_img).to(device)
    
    # 3. Tahmin (Inference)
    with torch.no_grad():
        prediction = model([img_tensor])[0]

    # 4. Sonuçları Filtrele (CPU'ya al)
    boxes = prediction['boxes'].cpu().numpy()
    scores = prediction['scores'].cpu().numpy()
    labels = prediction['labels'].cpu().numpy()
    
    # Eşik değerine göre eleme
    keep = scores >= SCORE_THRESHOLD
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]

    # 5. Görselleştirme (OpenCV ile çizim)
    img_np = np.array(original_img)
    img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

    for box, score, label_idx in zip(boxes, scores, labels):
        x1, y1, x2, y2 = map(int, box)
        
        # Etiket İsmini Bul
        # Modelin çıktısı "contiguous ID" (1,2,3), bunu gerçek isme çeviriyoruz
        # 'contig_to_cat_id' haritası önceki hücrelerden gelmeli
        if 'contig_to_cat_id' in globals() and 'id2name' in globals():
            cat_id = contig_to_cat_id.get(label_idx, label_idx)
            label_name = id2name.get(cat_id, str(label_idx))
        else:
            label_name = str(label_idx)

        # Kutu Çiz (Yeşil)
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Etiket Yaz
        text = f"{label_name}: {score:.2f}"
        (w_text, h_text), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
        
        # Yazı arka planı
        cv2.rectangle(img_bgr, (x1, y1 - 20), (x1 + w_text, y1), (0, 255, 0), -1)
        cv2.putText(img_bgr, text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1)

    # RGB'ye geri çevir ve göster
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    plt.subplot(NUM_IMAGES, 1, idx + 1)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f"Görsel: {img_info['file_name']} | Tespit Sayısı: {len(scores)}")

plt.tight_layout()
plt.show()